# Period Length Scans

This notebook fits a standard gaussian process with a fixed lengthscale and variance to the rate function of each of the cells, and scans over the period length to visualise the likelihood landscape of the data.

In [53]:
# Autoreloading of modules and files without restarting notebook.
%reload_ext autoreload
%autoreload 2

import GPy
import pickle
import numpy as np
from tqdm import tqdm
from pathlib import Path
from quantities import s
import matplotlib.pyplot as plt
from matplotlib.axes import Axes
from rvm_analysis.utils import get_test_times

#! Force the backend back to matplotlib inline (after importing viziphant).
print(plt.get_backend())
%matplotlib inline

module://matplotlib_inline.backend_inline


We try $100$ points over $\sim 25$ cells: $2500$ marginal likelihoods to evaluate for each cell group. Therefore $2500$ * $2$ ON and OFF + $2000$ Neutral = $7000$ to plot overall. Recompute is set to False so that the notebook can be run as a test, without performing the computation.

In [ ]:
recompute = False
save = False
save_path = Path("./PERIOD_SCOPE_RESULTS_200")
sampling_period=2*s
remove = 2
cut_time = None
neutral_cut = None
param_range=(0.1,1500)
num_points = 200
param_vals = np.linspace(param_range[0], param_range[1], num_points)

In [55]:
with open("../../CUT_CELL_ACTIVITIES.pkl", "rb") as file:
    cut_spikes_ON, cut_spikes_OFF, spikes_NEUTRAL, spikes_NEUTRAL_extra = pickle.load(file)

In [56]:
data = {
    "ON": get_test_times(cut_spikes_ON, kernel_width=2*s,sampling_period=sampling_period, cut_time=cut_time,remove=remove),
    "OFF": get_test_times(cut_spikes_OFF,kernel_width=2*s, sampling_period=sampling_period, cut_time=cut_time,remove=remove),
    "NEUTRAL": get_test_times(spikes_NEUTRAL, kernel_width=2*s,sampling_period=sampling_period, cut_time=neutral_cut,remove=remove),
    "NEUTRAL_EXTRA": get_test_times(spikes_NEUTRAL_extra,kernel_width=2*s, sampling_period=sampling_period, cut_time=neutral_cut,remove=remove)
    }

print(data['NEUTRAL'][0].shape)
print(data['NEUTRAL'][1].shape)
print(data['NEUTRAL'][2].shape)
print(data['NEUTRAL'][3].shape)

(20, 476)
(0,)
(476,)
(0,)


In [58]:
#! legacy function which also plots. Use the one coupled with the data saving function below.
def scan_param(X, Y,  kernel, param:str, ax: Axes, param_range=(0.1, 10), num_points=100,color=None):
    """
    Plots the log marginal likelihood of a Gaussian Process model with a periodic kernel
    as a function of the period parameter.

    Parameters:
    - X: numpy array of input data (shape: [n_samples, n_features]).
    - Y: numpy array of output data (shape: [n_samples, 1]).
    - param_range: tuple specifying the range of period values to explore (default: (0.1, 10)).
    - num_points: number of points in the period range to evaluate (default: 100).
    """
    # Generate a range of period values to evaluate
    param_vals = np.linspace(param_range[0], param_range[1], num_points)
    neg_log_marginal_likelihoods = []

    # Set up the GP model with a periodic kernel
    model = GPy.models.GPRegression(X, Y, kernel)

    # Evaluate the log marginal likelihood for each period value
    for p in param_vals:
        model[param]=p
        neg_log_marginal_likelihoods.append(-model.log_likelihood())

    # Plot the log marginal likelihood as a function of the period
    standardized = neg_log_marginal_likelihoods/np.max(neg_log_marginal_likelihoods)
    ax.plot(param_vals, standardized, label="NLML",color=color,linewidth=1)


In [59]:
def return_scan_param(X, Y, param: str, param_vals):
    """
    Returns the log marginal likelihood of a Gaussian Process model with a periodic kernel
    as a function of the period parameter.

    Parameters:
    - X: numpy array of input data (shape: [n_samples, n_features]).
    - Y: numpy array of output data (shape: [n_samples, 1]).
    - param_range: tuple specifying the range of period values to explore (default: (0.1, 10)).
    - num_points: number of points in the period range to evaluate (default: 100).
    """
    # Generate a range of period values to evaluate
    neg_log_marginal_likelihoods = np.empty((len(param_vals)))

    # Set up the GP model with a periodic kernel
    kernel = GPy.kern.StdPeriodic(input_dim=1, period=1.0,lengthscale=1,variance=5.0)  # Start with an arbitrary period
    model = GPy.models.GPRegression(X, Y, kernel)

    # Evaluate the log marginal likelihood for each period value
    for i, p in enumerate(param_vals):
        model[param]=p
        neg_log_marginal_likelihoods[i] = -model.log_likelihood()
    return neg_log_marginal_likelihoods

In [60]:
def return_iterated_log_likelihood_scan(data, param_vals, cell_type='ON',):
    """Returns the likelihood for a full set of cells as a `N_cells` x `num_points` matrix."""

    # Store the results
    N_cells = data[cell_type][0].shape[0]
    likelihoods = np.empty((*param_vals.shape,N_cells))

    X = data[cell_type][2]
    for i in tqdm(range(N_cells)):
        Y = data[cell_type][0][i]
        neg_log_liks = return_scan_param(X[:,None], Y[:,None],"std_periodic.period",param_vals)
        likelihoods[:,i] = neg_log_liks
        
    return likelihoods

In [ ]:
def run_likelihood_scans(save_path,save=False):

        ON_param_range_and_likelihoods = return_iterated_log_likelihood_scan(data,param_vals, cell_type='ON')
        OFF_param_range_and_likelihoods = return_iterated_log_likelihood_scan(data,param_vals, cell_type='OFF')
        NEUTRAL_param_range_and_likelihoods = return_iterated_log_likelihood_scan(data,param_vals, cell_type='NEUTRAL')
        NEUTRAL_extra_param_range_and_likelihoods = return_iterated_log_likelihood_scan(data,param_vals, cell_type='NEUTRAL_EXTRA')

        if save:
            np.savez_compressed(save_path,
                                param_values = param_vals,
                                ON=ON_param_range_and_likelihoods,
                                OFF=OFF_param_range_and_likelihoods,
                                NEUTRAL=NEUTRAL_param_range_and_likelihoods,
                                NEUTRAL_extra=NEUTRAL_extra_param_range_and_likelihoods,
                                )
        
if recompute:
    run_likelihood_scans(save_path,save=save)